# 语义记忆（Semantic Memory）

> **从对话中提取并存储泛化的事实，建立一个超越任何单次交互的持久知识库。**

你知道巴黎是法国的首都，但多半说不出自己是什么时候知道这件事的。这就是语义记忆在工作：它把经验蒸馏成稳定、与情境无关的事实。「什么时候学的」「怎么学的」都会淡忘，事实本身留了下来。

AI 智能体面对同样的挑战。用户在第一个会话里提到自己最喜欢的语言是 Python，第二个会话里提到团队规模，第三个会话里提到部署目标。没有语义记忆，每个会话都从零开始。有了语义记忆，智能体就能积累一份不断增长的、蒸馏后的事实档案。时间一长，它不再重复问同样的问题，还能根据已知信息预判需求。

本 notebook 教你用 **OpenAI SDK** 从零搭建一套语义记忆系统：用一次 LLM 调用从对话中提取事实，用向量嵌入（embedding，对语义的数值表示）存储事实，检测重复和矛盾，并为后续提示词检索相关事实。

**读完本 notebook 你会理解：**
- 如何从杂乱的对话文本中提取干净的陈述式事实。
- 嵌入相似度如何发现字符串匹配漏掉的重复事实。
- 当用户改变想法时，如何检测并解决矛盾。
- 语义记忆什么时候有用，什么时候会悄悄失效。

## 核心概念

- **语义记忆（semantic memory）**：存储一般性事实和知识的仓库，与具体情节分开。认知科学家 Endel Tulving 在 1972 年提出这个术语，用来区分「知道某件事」和「记得某段经历」。
- **事实提取（fact extraction）**：用 LLM 提示词从对话文本中抽取陈述式语句。「哦对了，我上个月换成 Mac 了」会变成事实「用户使用 macOS」。
- **嵌入（embedding）**：一串数字（向量），捕捉一段文本的语义。含义相近的文本，嵌入的方向也相近。我们用嵌入按语义比较事实，而不是按字面措辞。
- **余弦相似度（cosine similarity）**：衡量两个向量相似程度的指标，取值范围从 -1（方向相反）到 1（方向相同）。我们用它比较嵌入。
- **去重（deduplication）**：发现新事实和已存储的事实表达同一个意思。「我用 Mac」和「我的笔记本跑 macOS」是同一个事实。两者措辞不同，余弦相似度仍能识别出来。
- **矛盾检测（contradiction detection）**：发现新事实与已有事实冲突。「我用 Windows」与已存的「用户使用 macOS」矛盾。系统必须决定保留哪一个（通常保留较新的）。
- **置信度分数（confidence score）**：0.0 到 1.0 之间的数字，表示对一条事实的把握程度。明确陈述的事实置信度高，推断出的事实置信度低，反复提及会提升置信度。
- **上下文注入（context injection）**：把检索到的事实插入系统提示词，让 LLM 在生成回答时使用它们。

## 架构

<p align="center">
  <img src="../../images/diagrams/10_semantic_memory.svg" alt="架构图" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart LR
    Conv["Conversation"] --> Extractor["Fact Extractor\n(LLM call)"]
    Extractor --> DedupCheck["Dedup / Conflict\nChecker"]
    DedupCheck -- "new fact" --> KB["Knowledge Base\n(facts + confidence\n+ timestamps)"]
    DedupCheck -- "duplicate" --> Merge["Merge / Update\nConfidence"]
    Merge --> KB
    DedupCheck -- "contradiction" --> Resolve["Resolve Conflict\n(prefer recent)"]
    Resolve --> KB
    NewQuery["New Query"] --> Search["Semantic Search\n(embedding similarity)"]
    KB --> Search
    Search --> Injection["Context Injection"]
    Injection --> LLM["LLM"]
    LLM --> Response["Response"]
```

</details>

图中展示了两条主要流程。**写入路径（自左向右）：** 每次对话结束后，事实提取器（一次 LLM 调用）抽出陈述式事实。每条事实都要经过去重 / 冲突检查器，与已有知识库比对。新事实直接加入。重复事实被合并并提升置信度。矛盾通过保留较新的陈述来解决。

**读取路径（下方）：** 智能体需要回答时，先把当前查询转成嵌入，再在知识库中搜索相关事实，最后把它们注入提示词。LLM 看到的是过去所有会话积累下来的知识。

## 环境准备

安装依赖并配置 API 访问。聊天补全和嵌入都使用 OpenAI SDK。

In [ ]:
%pip install -q python-dotenv numpy deepagents langchain-openai

导入所有库，并从 `.env` 文件加载 API 密钥。

In [ ]:
import os
import json
import uuid
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
import numpy as np

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)
def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


网关没有 `/embeddings` 接口时，用这份本地确定性向量兜底：词加字符 n-gram 哈希后归一化。配置 `LLM_EMBEDDING_BASE_URL` 和 `LLM_EMBEDDING_MODEL_NAME` 后，自动切换到真实向量端点。


In [ ]:
EMBEDDING_MODEL = os.getenv("LLM_EMBEDDING_MODEL_NAME", "")


def embed_texts(texts: list[str]) -> list[list[float]]:
    """把文本转成向量。

    当前网关没有 /embeddings 接口。默认退回到本地确定性向量：
    词 + 字符 n-gram 哈希后归一化，语义相近的文本仍会得到相近向量，
    足够演示记忆检索的几何逻辑。配置 LLM_EMBEDDING_BASE_URL 和
    LLM_EMBEDDING_MODEL_NAME 后，自动改用真实向量端点。
    """
    if EMBEDDING_MODEL:
        from openai import OpenAI

        client = OpenAI(
            api_key=os.getenv("LLM_EMBEDDING_API_KEY", os.getenv("LLM_API_KEY")),
            base_url=os.getenv("LLM_EMBEDDING_BASE_URL", os.getenv("LLM_BASE_URL")),
        )
        response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
        return [item.embedding for item in response.data]

    import hashlib
    import math
    import re as re_mod

    dim = 512
    vectors = []
    for text in texts:
        vec = [0.0] * dim
        for word in re_mod.findall(r"\w+", text.lower()):
            grams = {word} | {word[i:i + 3] for i in range(max(1, len(word) - 2))}
            for gram in grams:
                digest = hashlib.blake2b(gram.encode("utf-8"), digest_size=8).digest()
                index = int.from_bytes(digest, "big") % dim
                vec[index] += 1.0
        norm = math.sqrt(sum(v * v for v in vec)) or 1.0
        vectors.append([v / norm for v in vec])
    return vectors


## 实现

我们分四部分搭建 `SemanticMemory` 类：

1. **事实数据结构**，外加一个计算余弦相似度的辅助函数。
2. **嵌入与提取**：把对话转化成带向量表示的事实。
3. **去重与矛盾检测**：把新事实与已有事实逐一比较。
4. **检索与聊天**：取出相关事实并注入提示词。

### 第 1 部分：事实数据结构

把它想象成一个档案柜，每张卡片记录一条事实。每张卡片上有事实本身、置信度分数、时间戳，以及用于搜索的嵌入。我们还需要一个快速衡量两张卡片相似程度的办法，那就是余弦相似度。

In [ ]:
@dataclass
class Fact:
    """从对话中提取出的一条知识。"""
    content: str                          # 例如「用户偏好深色模式」
    confidence: float = 0.8               # 取值 0.0 到 1.0
    fact_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    created_at: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
    last_confirmed: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
    mention_count: int = 1
    embedding: list[float] = field(default_factory=list)
    archived: bool = False                # 被更新的事实取代时置为 True


def cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
    """计算两个向量之间的余弦相似度。"""
    a = np.array(vec_a)
    b = np.array(vec_b)
    dot = np.dot(a, b)
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    if norm == 0:
        return 0.0
    return float(dot / norm)

### 第 2 部分：SemanticMemory 核心、嵌入与事实提取

`SemanticMemory` 类存储一个 `Fact` 对象列表。它用 OpenAI 嵌入 API 把文本转成向量，用一次聊天补全调用从对话消息中提取事实。

提取提示词是整个系统的核心。它要求 LLM 只抽出干净的陈述式语句，忽略寒暄、提问和假设。

In [ ]:
class SemanticMemory:
    """从对话中提取并存储事实的持久知识库。"""

    EXTRACTION_PROMPT = """You are a fact extractor. Read the conversation below and extract \
declarative facts about the user, their preferences, their work, or their world.

Rules:
- Output one fact per line.
- Each fact must be a short, standalone statement (under 20 words).
- Write in third person: "User ..." or "User's ...".
- Only extract facts that were clearly stated or strongly implied.
- Skip greetings, questions, opinions phrased as questions, and hypotheticals.
- If there are no extractable facts, output exactly: NONE

Conversation:
{conversation}

Extracted facts (one per line):"""

    CONTRADICTION_PROMPT = """Do these two facts contradict each other? \
Answer only YES or NO.

Fact A: {fact_a}
Fact B: {fact_b}

Answer:"""

    def __init__(
        self,
        similarity_threshold: float = 0.85,
        contradiction_threshold: float = 0.50,
    ):
        self.facts: list[Fact] = []
        self.similarity_threshold = similarity_threshold
        self.contradiction_threshold = contradiction_threshold

    # -- 嵌入 ------------------------------------------------------------
    def _embed(self, text: str) -> list[float]:
        """获取一段文本的嵌入向量。"""
        return embed_texts([text])[0]

    def _embed_batch(self, texts: list[str]) -> list[list[float]]:
        """一次 API 调用嵌入多段文本。"""
        return embed_texts(texts)
        return response



`extract_facts` 方法把对话消息连同一段精心设计的提示词一起发给 LLM。提示词要求模型只提取清晰的陈述式语句，跳过寒暄、提问和假设。每条提取出的事实都是一个简短、独立的句子。

In [ ]:
# -- 事实提取 ----------------------------------------------------------
def _extract_facts(self, messages: list[dict]) -> list[str]:
    """用一次 LLM 调用从消息中提取陈述式事实。"""
    conversation_text = "\n".join(
        f"{m['role'].upper()}: {m['content']}" for m in messages
    )
    response = llm.invoke([{
            "role": "user",
            "content": self.EXTRACTION_PROMPT.format(conversation=conversation_text),
        }])
    raw = to_text(response.content).strip()
    if raw == "NONE":
        return []
    # 每行一条事实
    facts = [line.strip().lstrip("- ") for line in raw.splitlines() if line.strip()]
    return facts

SemanticMemory.extract_facts = _extract_facts


### 第 3 部分：去重与矛盾检测

想象两个人用不同的措辞登记同一条客户投诉。好的柜员能看出它们说的是同一件事。这就是去重。再想象一位客户一月说「我爱这个产品」，三月说「我讨厌这个产品」。柜员需要标记矛盾，并保留较新的那条。

这两件事都在 `_find_match` 里完成。对每条新事实，我们计算它的嵌入，并与所有已存事实比较。相似度很高（高于 0.85）说明是重复；相似度中等（高于 0.50）说明两条事实话题相同，这时我们让 LLM 判断它们是否互相矛盾。

In [ ]:
def _find_match(self, new_text: str, new_embedding: list[float]):
    """检查新事实是重复已有事实，还是与已有事实矛盾。

    返回：
        ("duplicate", existing_fact)：事实已存在。
        ("contradiction", existing_fact)：与已存事实冲突。
        ("new", None)：全新的事实。
    """
    best_sim = 0.0
    best_fact = None

    for fact in self.facts:
        if fact.archived:
            continue
        sim = cosine_similarity(new_embedding, fact.embedding)
        if sim > best_sim:
            best_sim = sim
            best_fact = fact

    if best_sim >= self.similarity_threshold:
        return ("duplicate", best_fact)

    if best_sim >= self.contradiction_threshold and best_fact is not None:
        # 话题相同但内容不同。让 LLM 判断是否矛盾。
        if self._check_contradiction(best_fact.content, new_text):
            return ("contradiction", best_fact)

    return ("new", None)


def _check_contradiction(self, fact_a: str, fact_b: str) -> bool:
    """询问 LLM 两条事实是否互相矛盾。"""
    response = llm.invoke([{
            "role": "user",
            "content": self.CONTRADICTION_PROMPT.format(fact_a=fact_a, fact_b=fact_b),
        }])
    answer = to_text(response.content).strip().upper()
    return answer.startswith("YES")


# 把方法挂到类上
SemanticMemory._find_match = _find_match
SemanticMemory._check_contradiction = _check_contradiction

### 第 4 部分：处理、检索与聊天

现在把所有部件连起来。`process_conversation` 从消息中提取事实，并把每条事实分流到新增、合并或冲突解决。`retrieve` 为查询找出最相关的事实。`chat` 把一切串起来：检索相关知识、注入系统提示词、调用 LLM。

In [ ]:
def process_conversation(self, messages: list[dict]) -> dict:
    """从消息中提取事实并更新知识库。

    返回处理结果的统计：新增、合并和解决矛盾的事实数量。
    """
    extracted = self.extract_facts(messages)
    if not extracted:
        return {"extracted": 0, "new": 0, "merged": 0, "contradictions": 0}

    embeddings = self._embed_batch(extracted)
    stats = {"extracted": len(extracted), "new": 0, "merged": 0, "contradictions": 0}

    for fact_text, embedding in zip(extracted, embeddings):
        match_type, existing = self._find_match(fact_text, embedding)

        if match_type == "new":
            self.facts.append(Fact(content=fact_text, embedding=embedding))
            stats["new"] += 1

        elif match_type == "duplicate":
            # 提升置信度并更新时间戳
            existing.mention_count += 1
            existing.confidence = min(1.0, existing.confidence + 0.05)
            existing.last_confirmed = datetime.now(timezone.utc).isoformat()
            stats["merged"] += 1

        elif match_type == "contradiction":
            # 归档旧事实，加入新事实
            existing.archived = True
            self.facts.append(Fact(
                content=fact_text,
                embedding=embedding,
                confidence=0.9,  # 较新的陈述获得较高置信度
            ))
            stats["contradictions"] += 1

    return stats




`retrieve` 方法用余弦相似度找出与查询相关的已存事实。`chat` 方法把一切串起来：检索相关事实、注入系统提示词、调用 LLM。`get_active_facts` 返回所有未归档的事实，便于检查。

In [ ]:
def retrieve(self, query: str, top_k: int = 5) -> list[tuple[Fact, float]]:
    """用余弦相似度找出与查询最相关的事实。"""
    query_embedding = self._embed(query)
    scored = []
    for fact in self.facts:
        if fact.archived:
            continue
        sim = cosine_similarity(query_embedding, fact.embedding)
        scored.append((fact, sim))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]


def chat(self, user_input: str, conversation_history: list[dict] | None = None) -> str:
    """在提示词中注入相关事实后回答用户。"""
    relevant = self.retrieve(user_input, top_k=5)

    # 用已知事实构建系统提示词
    system_parts = ["You are a helpful assistant."]
    if relevant:
        facts_text = "\n".join(
            f"- {fact.content} (confidence: {fact.confidence:.2f})"
            for fact, _sim in relevant
        )
        system_parts.append(
            f"\nHere is what you know about the user:\n{facts_text}\n"
            "Use these facts naturally. Don't list them unless asked."
        )

    messages = list(conversation_history or [])
    messages.append({"role": "user", "content": user_input})

    # 事实随对话更新，系统提示词每轮都变，所以每轮重建智能体
    agent = create_deep_agent(
        model=llm,
        system_prompt="\n".join(system_parts),
    )
    result = agent.invoke({"messages": messages})
    return to_text(result["messages"][-1].content)


def get_active_facts(self) -> list[Fact]:
    """返回所有未归档的事实。"""
    return [f for f in self.facts if not f.archived]


# 把方法挂到类上
SemanticMemory.process_conversation = process_conversation
SemanticMemory.retrieve = retrieve
SemanticMemory.chat = chat
SemanticMemory.get_active_facts = get_active_facts

## 运行示例

我们来走一个贴近真实的场景。用户与智能体聊了三个会话：第一个会话互相认识，第二个会话聊工作细节，第三个会话用户改变了偏好。我们会看到知识库如何增长、去重，并解决一次矛盾。

### 会话 1：认识用户

用户分享基本信息，事实提取器从中抽出陈述式语句。

In [ ]:
memory = SemanticMemory()

session_1 = [
    {"role": "user", "content": "Hi! I'm Maya. I'm a data scientist living in Berlin."},
    {"role": "assistant", "content": "Nice to meet you, Maya! Berlin is a great city for tech. What are you working on?"},
    {"role": "user", "content": "I'm building a recommendation engine for an e-commerce company. We use Python and PyTorch mostly."},
    {"role": "assistant", "content": "Sounds like a solid stack. How large is your dataset?"},
    {"role": "user", "content": "About 50 million user interactions. We store everything in PostgreSQL and train on AWS."},
]

stats = memory.process_conversation(session_1)
print("Session 1 results:", stats)
print(f"\nKnowledge base now has {len(memory.get_active_facts())} facts:")
for fact in memory.get_active_facts():
    print(f"  [{fact.confidence:.2f}] {fact.content}")

### 会话 2：更多信息（含重复）

用户再次提到一些相同的信息。系统应该合并这些重复事实并提升置信度，而不是新建条目。

In [ ]:
session_2 = [
    {"role": "user", "content": "I'm back! So we talked about my recommendation project before."},
    {"role": "assistant", "content": "Welcome back, Maya! Yes, the e-commerce recommendation engine. How's it going?"},
    {"role": "user", "content": "Good! My team has 5 engineers. We deploy our models on AWS SageMaker."},
    {"role": "assistant", "content": "SageMaker is a nice fit for that. Any challenges?"},
    {"role": "user", "content": "Latency is our biggest issue. We need sub-100ms inference for real-time recommendations."},
]

stats = memory.process_conversation(session_2)
print("Session 2 results:", stats)
print(f"\nKnowledge base now has {len(memory.get_active_facts())} active facts:")
for fact in memory.get_active_facts():
    confirmed = "(boosted)" if fact.mention_count > 1 else ""
    print(f"  [{fact.confidence:.2f}] {fact.content} {confirmed}")

### 会话 3：一次矛盾

用户搬了家。这与已存储的事实矛盾。系统应该归档旧事实（「住在柏林」），存储新事实（「住在阿姆斯特丹」）。

In [ ]:
session_3 = [
    {"role": "user", "content": "Hey, big news! I moved to Amsterdam last month for a new role."},
    {"role": "assistant", "content": "Congratulations on the move! What's the new role?"},
    {"role": "user", "content": "I'm now a senior ML engineer at a fintech startup. Still using Python though!"},
]

stats = memory.process_conversation(session_3)
print("Session 3 results:", stats)
print(f"\nActive facts ({len(memory.get_active_facts())}):")
for fact in memory.get_active_facts():
    print(f"  [{fact.confidence:.2f}] {fact.content}")

archived = [f for f in memory.facts if f.archived]
if archived:
    print(f"\nArchived facts ({len(archived)}):")
    for fact in archived:
        print(f"  [archived] {fact.content}")

### 检索：找出相关事实

用户提问时，系统会检索与该查询相关的事实。我们用几个不同的查询试一试。

In [ ]:
test_queries = [
    "What tech stack does Maya use?",
    "Where does the user live?",
    "Tell me about the user's team.",
]

for query in test_queries:
    print(f"Query: {query}")
    results = memory.retrieve(query, top_k=3)
    for fact, score in results:
        print(f"  [{score:.3f}] {fact.content}")
    print()

### 带着记忆聊天

现在看完整闭环。智能体利用检索到的事实，回答本会话中从未被直接问过的问题。

In [ ]:
questions = [
    "Can you recommend a good deep learning conference near me?",
    "What programming language should I use for my next project?",
    "What do you know about me?",
]

for q in questions:
    print(f"User: {q}")
    reply = memory.chat(q)
    print(f"Agent: {reply}\n")

### 持久化

重启就消失的语义记忆没有用处。我们把知识库序列化成 JSON，让它在会话之间保存下来。

In [ ]:
def save_memory(mem: SemanticMemory, path: str) -> None:
    """把知识库保存为 JSON 文件。"""
    data = [asdict(f) for f in mem.facts]
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    active = len(mem.get_active_facts())
    print(f"Saved {active} active facts to {path}")


def load_memory(path: str, **kwargs) -> SemanticMemory:
    """从 JSON 文件加载知识库。"""
    with open(path) as f:
        data = json.load(f)
    mem = SemanticMemory(**kwargs)
    for item in data:
        mem.facts.append(Fact(**item))
    active = len(mem.get_active_facts())
    print(f"Loaded {active} active facts from {path}")
    return mem


# 读写回环测试
save_memory(memory, "semantic_memory.json")
loaded = load_memory("semantic_memory.json")

# 验证加载后的记忆可用
reply = loaded.chat("Where do I live now?")
print(f"\nAgent (from loaded memory): {reply}")

清理临时文件。

In [ ]:
if os.path.exists("semantic_memory.json"):
    os.remove("semantic_memory.json")

## 权衡

### 语义记忆表现良好的场景

- **长期存活的智能体。** 如果你的智能体在多个会话里服务同一位用户，语义记忆会积累出丰富的用户画像。用户不必反复自我介绍，智能体也显得更懂人。
- **跨会话知识。** 一月提取的事实，七月仍然可用。对话缓冲记忆和滑动窗口记忆在会话结束时就丢掉了这些信息。
- **高效利用上下文。** 不必重发几百条历史消息，只需注入 5 到 10 条相关事实。这既省 token，又让提示词保持聚焦。

### 什么时候会失效

- **提取错误。** LLM 可能把观点当成事实提取，漏掉隐含信息，甚至捏造没人说过的事实。知识库里的每条事实都必须真的被说过。
- **微妙矛盾。** 「我远程办公」和「我每周二去办公室」并不严格矛盾，但系统可能处理不好这种细微差别。
- **扩展性。** 每条新事实都要与所有已存事实比较，复杂度是 O(n)。事实达到数千条时，你需要近似最近邻搜索（ANN），而不是暴力的余弦相似度比较。
- **陈旧事实。** 没有时间衰减机制（随时间降低置信度），过时的事实会一直留着。用户可能半年前就换了工作，旧职位却还在。

### 成本考量

每个对话轮次需要一次 LLM 调用做提取，每条提取出的事实需要一次嵌入调用，矛盾检查可能再加一次 LLM 调用。一个产出 3 条事实的轮次，大约产生 2 到 5 次 API 调用。按轮次算，这比对话缓冲记忆贵；但会话较长时，按会话算反而更便宜。

## 延伸阅读

- [Tulving, "Episodic and Semantic Memory," *Organization of Memory*, 1972](https://psycnet.apa.org/record/1973-08477-005) - Endel Tulving 的原始章节，他首次把语义记忆定义为独立于情景记忆（episodic memory）的系统。
- [Park et al., "Generative Agents: Interactive Simulacra of Human Behavior," UIST 2023 (arXiv:2304.03442)](https://arxiv.org/abs/2304.03442) - 在模拟智能体中同时演示了情景记忆和语义记忆，这些智能体会从经验中形成信念。
- [OpenAI Embeddings Guide](https://platform.openai.com/docs/guides/embeddings) - 本 notebook 所用嵌入 API 的官方文档。
- [Mem0: Long-Term Memory for AI Agents](https://github.com/mem0ai/mem0?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - 实现了事实提取、去重和矛盾检测的开源库。
- [Zep: Long-Term Memory for AI Assistants](https://github.com/getzep/zep?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - 一个记忆服务器，从对话中提取事实，并对积累的知识提供语义搜索。

---

*上一篇：[09 - 情景记忆](../09_episodic_memory/) | 下一篇：[11 - 程序性记忆](../11_procedural_memory/) ->*

## 🧪 动手试一试

三个小挑战，帮你加深理解。每个大约需要 10 到 30 分钟。

### 挑战 1：阈值敏感度
把重复检测阈值（默认 0.85）分别调成 0.75、0.80、0.90 和 0.95。每种设置下处理同一段 20 轮对话，统计有多少事实被判为新增、重复或矛盾，并为每个阈值画出混淆矩阵。

### 挑战 2：事实准确性审计
处理完 20 轮对话后，用 `get_active_facts()` 导出所有活跃事实。人工把每条事实标注为正确、过时或错误，计算准确率，并判断错误来自提取、去重还是矛盾解决。

### 挑战 3：分类事实库
给 `Fact` 数据类加一个 `category` 字段，把事实分进命名类别（如 'personal'、'professional'、'preferences'）。参考技术 17 记忆路由中的分类方法，把事实路由到对应类别。比较在单个类别内检索与在全部事实中检索的质量差异。

![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--10-semantic-memory--semantic-memory)
